# Installation


In [ ]:
!pip install rouge-score pycocoevalcap

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 7.2 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=ec184c1bb8155e36619d607e7704f8e9e48844105e7b852ace62826fd677ebef
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [ ]:
!pip install torchxrayvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 66.0 MB/s eta 0:00:00


# Lib


In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
from typing import List

import os
import re
import ast
import math
import glob
import copy
import json
import random
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import kagglehub
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from torch.optim.lr_scheduler import CosineAnnealingLR

import torchxrayvision as xrv

from transformers import AutoTokenizer, AutoModel

import nltk

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from pycocoevalcap.cider.cider import Cider
from rouge_score import rouge_scorer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Data Preparation


## Vocabulary

`Vocabulary` có nhiệm vụ:
- Xây dựng từ điển từ tập văn bản huấn luyện.
- Ánh xạ từ -> số nguyên (`stoi`).
- Ánh xạ số nguyên -> từ (`itos`).
- Chuyển câu thành chuỗi ID để mô hình xử lý.
- Chuyển kết quả dự đoán từ ID về văn bản.

Special tokens:
- `PAD_TOKEN = "<PAD>"` # Bổ sung vào cuối câu để các câu có cùng độ dài
- `SOS_TOKEN = "<SOS>"` # Đánh dấu bắt đầu câu
- `EOS_TOKEN = "<EOS>"` # Đánh đâu kết thúc câu
- `UNK_TOKEN = "<UNK>"` # Đại diện cho từ không xuất hiện trong từ điển


In [ ]:
class Vocabulary:
    PAD_TOKEN = "<PAD>"
    SOS_TOKEN = "<SOS>"
    EOS_TOKEN = "<EOS>"
    UNK_TOKEN = "<UNK>"

    PAD = 0
    SOS = 1
    EOS = 2
    UNK = 3

    def __init__(self, freq_threshold: int = 3):
        self.freq_threshold = freq_threshold

        self.itos = {
            self.PAD: self.PAD_TOKEN,
            self.SOS: self.SOS_TOKEN,
            self.EOS: self.EOS_TOKEN,
            self.UNK: self.UNK_TOKEN,
        }

        self.stoi = {v: k for k, v in self.itos.items()}

    def __len__(self) -> int:
        return len(self.itos)

    def build_vocabulary(self, corpus: List[str]) -> None:
        frequencies = Counter()
        idx = len(self.itos)

        for sentence in corpus:
            if not isinstance(sentence, str):
                continue

            for token in sentence.lower().split():
                frequencies[token] += 1

                if frequencies[token] >= self.freq_threshold and token not in self.stoi:
                    self.stoi[token] = idx
                    self.itos[idx] = token
                    idx += 1

    def numericalize(self, text: str) -> List[int]:
        if not isinstance(text, str):
            return []

        return [self.stoi.get(token, self.UNK) for token in text.lower().split()]

    def decode(
        self,
        ids: List[int],
        skip_special: bool = True,
    ) -> str:

        special_tokens = {
            self.PAD,
            self.SOS,
            self.EOS,
        }

        words = []

        for idx in ids:

            if skip_special and idx in special_tokens:
                continue

            words.append(self.itos.get(idx, self.UNK_TOKEN))

        return " ".join(words)

## Dataset custom

`MIMICCXRDataset` dùng để:
- Đọc ảnh X-ray từ thư mục lưu trữ.
- Đọc báo cáo (caption) tương ứng.
- Tiền xử lý ảnh.
- Chuyển văn bản thành chuỗi ID.
- Trả về dữ liệu để `DataLoader` tạo batch huấn luyện.

In [ ]:
class MIMICCXRDataset(Dataset):
    def __init__(
        self,
        df,
        root_dir: str,
        vocab: Vocabulary,
        transform=None,
        max_length: int = 120,
    ):
        self.records = df.to_dict("records")

        self.root_dir = root_dir
        self.vocab = vocab
        self.transform = transform
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int):

        row = self.records[index]

        caption = row["text"]

        image_rel_path = row["best_image"]

        image_path = os.path.join(
            self.root_dir,
            image_rel_path,
        )

        if not os.path.exists(image_path):
            raise FileNotFoundError(image_path)

        img = Image.open(image_path).convert("L")

        if self.transform is not None:
            img = self.transform(img)

        tokens = self.vocab.numericalize(caption)

        tokens = tokens[: self.max_length - 2]

        caption_ids = [self.vocab.SOS] + tokens + [self.vocab.EOS]

        caption_tensor = torch.tensor(
            caption_ids,
            dtype=torch.long,
        )

        return (
            img,
            caption_tensor,
            image_path,
        )

## Collate

`MyCollate` được sử dụng để:
- Ghép nhiều ảnh thành một batch tensor.
- Padding các caption có độ dài khác nhau.
- Giữ lại đường dẫn ảnh để phục vụ debug hoặc visualization.


In [ ]:
class MyCollate:
    def __init__(self, pad_idx: int):
        self.pad_idx = pad_idx

    def __call__(self, batch):

        images = torch.stack(
            [sample[0] for sample in batch],
            dim=0,
        )

        captions = pad_sequence(
            [sample[1] for sample in batch],
            batch_first=True,
            padding_value=self.pad_idx,
        )

        paths = [sample[2] for sample in batch]

        return (
            images,
            captions,
            paths,
        )

# Func


## Utils

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = text.replace("findings:", "")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def print_directory_structure(startpath, max_level=3):
    start_level = startpath.count(os.sep)
    for root, dirs, files in os.walk(startpath):
        level = root.count(os.sep) - start_level
        if level > max_level:
            del dirs[:]
            continue
        indent = ' ' * 4 * (level)
        print(f'{indent}{os.path.basename(root)}/')
        if level < max_level:
            subindent = ' ' * 4 * (level + 1)
            for f in files:
                print(f'{subindent}{f}')

# Load data

### Dowload

In [ ]:
DATA_DIR = kagglehub.dataset_download("simhadrisadaram/mimic-cxr-dataset")
print("Path to dataset files:", DATA_DIR)

100%|██████████| 16.5G/16.5G [03:32<00:00, 83.3MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/simhadrisadaram/mimic-cxr-dataset/versions/2


In [ ]:
split_dir = kagglehub.dataset_download("avcuongy/mimic-cxr-split")
print("Path to dataset files:", split_dir)

100%|██████████| 4.44M/4.44M [00:00<00:00, 151MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/avcuongy/mimic-cxr-split/versions/1


In [ ]:
ROOT_DIR = os.path.join(DATA_DIR, "official_data_iccv_final")
TRAIN_CSV = os.path.join(split_dir, "mimic_train.csv")
VAL_CSV = os.path.join(split_dir, "mimic_val.csv")
TEST_CSV = os.path.join(split_dir, "mimic_test.csv")

print(f"DATA_DIR: {DATA_DIR}")
print(f"TRAIN CSV: {TRAIN_CSV}")
print(f"VAL CSV: {VAL_CSV}")
print(f"TEST CSV: {TEST_CSV}")

DATA_DIR: /root/.cache/kagglehub/datasets/simhadrisadaram/mimic-cxr-dataset/versions/2
TRAIN CSV: /root/.cache/kagglehub/datasets/avcuongy/mimic-cxr-split/versions/1/mimic_train.csv
VAL CSV: /root/.cache/kagglehub/datasets/avcuongy/mimic-cxr-split/versions/1/mimic_val.csv
TEST CSV: /root/.cache/kagglehub/datasets/avcuongy/mimic-cxr-split/versions/1/mimic_test.csv


## Calculation

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print(f"{train_df.shape}")
print(f"{val_df.shape}")
print(f"{test_df.shape}")

In [ ]:
for df in [train_df, val_df, test_df]:
    df["text"] = df["text"].fillna("").apply(clean_text)

In [ ]:
freq_threshold = 7
vocab = Vocabulary(freq_threshold=freq_threshold)
vocab.build_vocabulary(train_df["text"].tolist())
print(f"Vocabulary size: {len(vocab)}")
my_collate = MyCollate(pad_idx=vocab.PAD)

In [ ]:
IMG_SIZE = 256
batch_size = 32
num_workers = 4

calc_transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
        transforms.RandomCrop(IMG_SIZE),
        transforms.ColorJitter(
            brightness=0.2,
            contrast=0.2,
        ),
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
        transforms.ToTensor(),
    ]
)

temp_train_dataset = MIMICCXRDataset(df=train_df, root_dir=ROOT_DIR, transform=calc_transform, vocab=vocab)
temp_train_loader = DataLoader(
    dataset=temp_train_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    shuffle=False,
    collate_fn=my_collate
)

sum_of_pixels = 0.0
sum_of_squared_pixels = 0.0
total_num_pixels = 0

for images, _, _ in tqdm(temp_train_loader, desc="Calculating mean and std of train_dataset"):
    sum_of_pixels += torch.sum(images)
    sum_of_squared_pixels += torch.sum(images**2)
    total_num_pixels += images.numel()

calculated_mean = sum_of_pixels / total_num_pixels
calculated_std = torch.sqrt((sum_of_squared_pixels / total_num_pixels) - (calculated_mean**2))

print(f"Calculated Mean of train_dataset: {calculated_mean.item():.4f}")
print(f"Calculated Std of train_dataset: {calculated_std.item():.4f}")

Calculating mean and std of train_dataset: 100%|██████████| 1563/1563 [04:55<00:00,  5.29it/s]


Calculated Mean of train_dataset: 0.4888
Calculated Std of train_dataset: 0.2839
